# 04 - PostgreSQL

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Service: `postgres:5432`. StatefulSet/pod: `postgres` / `postgres-0`. PVC: `postgres-data`, 5 GiB. Container volume name: `data`; database files use `/var/lib/postgresql/data`. Initial database: `polaris`, account: `team`. Polaris catalog metadata will use this database. Example database: `test_database`; table: `test_table`.

## Apply PostgreSQL

```bash
kubectl apply -f postgresql/postgresql.yaml
```



## Wait for PostgreSQL

```bash
kubectl rollout status statefulset/postgres --timeout=600s
```



## Inspect PostgreSQL storage

```bash
kubectl get pod postgres-0
kubectl get service postgres
kubectl get pvc postgres-data
```



## Authenticate and inspect the database

```bash
kubectl exec postgres-0 -- sh -ec 'export PGPASSWORD="$POSTGRES_PASSWORD"; psql -h postgres -U "$POSTGRES_USER" -d "$POSTGRES_DB" -v ON_ERROR_STOP=1 -c "SELECT current_database(), current_user, version();"'
```



## Create test_database

```bash
kubectl exec postgres-0 -- sh -ec 'export PGPASSWORD="$POSTGRES_PASSWORD"; if ! psql -h postgres -U "$POSTGRES_USER" -d postgres -Atc "SELECT datname FROM pg_database WHERE datname='\''test_database'\''" | grep -qx test_database; then createdb -h postgres -U "$POSTGRES_USER" test_database; fi'
```



## Create and write test_table

```bash
kubectl exec -i postgres-0 -- sh -ec 'export PGPASSWORD="$POSTGRES_PASSWORD"; exec psql -h postgres -U "$POSTGRES_USER" -d test_database -v ON_ERROR_STOP=1' <<'SQL'
CREATE TABLE IF NOT EXISTS test_table (id INTEGER PRIMARY KEY, message TEXT NOT NULL);
INSERT INTO test_table VALUES (1, 'workspace database example') ON CONFLICT (id) DO UPDATE SET message = EXCLUDED.message;
SELECT * FROM test_table;
SQL
```

Expect row 1 and `workspace database example`. This runs psql inside the pod; no host database client or Docker exec is needed.

## Check first-start logs and metrics

```bash
python3 validations/check-observability.py --components postgres
```

Monitoring was started in notebook 02. This checks the actual first-start daemon logs and CPU/memory samples without restarting the service to generate logs.


After Polaris is deployed, stop it before PostgreSQL. `bash cluster.sh stop postgres` handles this. The next two commands are for this stage before Polaris is deployed, or after stopping Polaris.

## Stop this service

```bash
kubectl scale statefulset/postgres --replicas=0
kubectl wait --for=delete pod -l app=postgres --timeout=180s
```

This retains its Service and storage. Run after finishing the examples.

## Resume this service

```bash
kubectl scale statefulset/postgres --replicas=1
kubectl rollout status statefulset/postgres --timeout=600s
```

